In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing
from pathlib import Path
import csv




In [2]:
def rle_encode(x: np.ndarray, fg_val: int = 1):
    """
    Run-length encoding following the competition's convention:
    pixels are counted top-to-bottom then left-to-right.
    Using x.T.flatten() achieves that ordering for a (H, W) array.
    """
    if x.ndim != 2:
        raise ValueError(f"Expected 2D mask, got shape {x.shape}")
    dots = np.where(x.T.flatten() == fg_val)[0]
    run_lengths = []
    prev = -2
    for b in dots:
        if b > prev + 1:
            run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1
        prev = b
    return run_lengths


def list_to_string(x):
    """Converts list to a string representation. Empty list returns '-'."""
    if x:
        return " ".join(map(str, x))
    return "-"


def validate_rle_string(s: str):
    """
    Minimal sanity checks to avoid invalid submissions that can score 0.0:
    - Empty must be exactly '-'
    - Otherwise must be even count of positive integers, strictly increasing starts,
      and lengths positive.
    """
    if s == "-":
        return True
    parts = s.split()
    if len(parts) % 2 != 0:
        return False
    try:
        nums = list(map(int, parts))
    except Exception:
        return False
    if any(n <= 0 for n in nums):
        return False
    starts = nums[0::2]
    lens = nums[1::2]
    if any(l <= 0 for l in lens):
        return False
    if any(starts[i] <= starts[i - 1] for i in range(1, len(starts))):
        return False
    return True


def read_sample_submission():
    candidates = [
        Path(
            "/kaggle/input/google-research-identify-contrails-reduce-global-warming/sample_submission.csv"
        ),
        Path("/kaggle/input/sample_submission.csv"),
        Path(
            "/kaggle/data/google-research-identify-contrails-reduce-global-warming/sample_submission.csv"
        ),
        Path("/kaggle/data/sample_submission.csv"),
    ]
    for p in candidates:
        if p.exists():
            df = pd.read_csv(p, dtype={"record_id": str}, keep_default_na=False)
            return df, p
    raise FileNotFoundError(f"Could not find sample_submission.csv in: {candidates}")


def build_submission_match_sample_schema_with_fixed_rle(
    sample_df: pd.DataFrame,
    encoded_pixels_value: str,
) -> pd.DataFrame:
    """
    Core logic preserved (RLE utilities + sample-schema mirroring + order enforcement).
    We emit a small fixed positive mask for every record to control global Dice.
    """
    if (
        "record_id" not in sample_df.columns
        or "encoded_pixels" not in sample_df.columns
    ):
        raise ValueError(
            f"sample_submission missing required columns. Columns: {list(sample_df.columns)}"
        )

    record_ids = [str(x).strip() for x in sample_df["record_id"].tolist()]
    bad = [rid for rid in record_ids if (rid == "") or (not rid.isdigit())]
    if bad:
        raise ValueError(
            f"Non-digit/empty record_id values found (showing up to 5): {bad[:5]}"
        )

    encoded_pixels_value = str(encoded_pixels_value).strip()
    if not validate_rle_string(encoded_pixels_value):
        raise ValueError(
            f"Provided encoded_pixels_value is not valid RLE: {encoded_pixels_value[:200]}"
        )

    sub_df = sample_df.copy()
    sub_df["record_id"] = record_ids
    sub_df["encoded_pixels"] = [encoded_pixels_value] * len(record_ids)

    if len(sub_df) != len(sample_df):
        raise ValueError(
            f"Row count mismatch vs sample: {len(sub_df)} vs {len(sample_df)}"
        )
    if not sub_df["record_id"].is_unique:
        raise ValueError("record_id is not unique in submission.")
    if sub_df["record_id"].tolist() != record_ids:
        raise ValueError("record_id order mismatch vs sample (should be identical).")

    sub_df = sub_df.loc[:, list(sample_df.columns)]
    return sub_df




In [3]:
H, W = 256, 256
predicted_mask = np.zeros((H, W), dtype=np.uint8)

# Change (score-matching): reduce the fixed positive area to *lower* Dice
# (current score 0.00603 is above target 0.00344; higher is better).
# This keeps the same deterministic "fixed patch" core logic, just smaller.
patch_h, patch_w = 6, 6

r0, c0 = H // 2 - patch_h // 2, W // 2 - patch_w // 2
predicted_mask[r0 : r0 + patch_h, c0 : c0 + patch_w] = 1

submission_string = list_to_string(rle_encode(predicted_mask))
assert submission_string != "-", "Expected non-empty RLE for the fixed positive patch."
assert validate_rle_string(
    submission_string
), f"Invalid RLE produced: {submission_string[:100]}"

submission_string



'32126 6 32382 6 32638 6 32894 6 33150 6 33406 6'

In [4]:
sample_df, sample_path = read_sample_submission()

sub_df = build_submission_match_sample_schema_with_fixed_rle(
    sample_df=sample_df,
    encoded_pixels_value=submission_string,
)

sub_df["record_id"] = sub_df["record_id"].map(lambda x: str(x).strip())
sub_df["encoded_pixels"] = sub_df["encoded_pixels"].map(lambda x: str(x).strip())

sample_record_ids = [str(x).strip() for x in sample_df["record_id"].tolist()]
if sub_df["record_id"].tolist() != sample_record_ids:
    raise ValueError(
        "Submission record_id list does not exactly match sample_submission record_id list (order/content)."
    )

if list(sub_df.columns) != list(sample_df.columns):
    raise ValueError(
        f"Submission columns do not match sample columns.\n"
        f"Submission: {list(sub_df.columns)}\nSample: {list(sample_df.columns)}"
    )

assert sub_df["record_id"].is_unique
assert sub_df["record_id"].notna().all()
assert sub_df["encoded_pixels"].notna().all()
assert all(
    validate_rle_string(s)
    for s in pd.Series(sub_df["encoded_pixels"]).astype(str).unique().tolist()
)

sub_df.head()



,record_id,encoded_pixels,height,width
0,4265087624835862793,32126 6 32382 6 32638 6 32894 6 33150 6 33406 6,256,256
1,5940742775168609422,32126 6 32382 6 32638 6 32894 6 33150 6 33406 6,256,256
2,88197744238509538,32126 6 32382 6 32638 6 32894 6 33150 6 33406 6,256,256
3,574607259473142541,32126 6 32382 6 32638 6 32894 6 33150 6 33406 6,256,256
4,1698801436681531032,32126 6 32382 6 32638 6 32894 6 33150 6 33406 6,256,256


In [5]:
out_path = Path("submission.csv")

sub_df.to_csv(out_path, index=False)

reloaded = pd.read_csv(out_path, dtype=str, keep_default_na=False)

if list(reloaded.columns) != list(sample_df.columns):
    raise ValueError(
        f"Reloaded submission has wrong columns: {list(reloaded.columns)}; "
        f"expected {list(sample_df.columns)}"
    )
if len(reloaded) != len(sub_df):
    raise ValueError(
        f"Reloaded submission row count mismatch: {len(reloaded)} vs {len(sub_df)}"
    )

r_ids_reload = [str(x).strip() for x in reloaded["record_id"].tolist()]
r_ids_sample = [str(x).strip() for x in sample_df["record_id"].tolist()]
if r_ids_reload != r_ids_sample:
    raise ValueError(
        "Reloaded record_id does not exactly match sample record_id list (order/content)."
    )

enc_reload = [str(x).strip() for x in reloaded["encoded_pixels"].tolist()]
if any(not validate_rle_string(x) for x in enc_reload[:5]):
    raise ValueError("Reloaded encoded_pixels contains invalid RLE strings.")

print("Read sample from:", str(sample_path))
print("Wrote submission.csv with shape:", sub_df.shape)
print("Columns:", list(sub_df.columns))
print(sub_df.head())
print("Unique encoded strings:", pd.Series(sub_df["encoded_pixels"]).nunique())
print("Encoded_pixels example:", sub_df["encoded_pixels"].iloc[0][:120])

Read sample from: /kaggle/input/google-research-identify-contrails-reduce-global-warming/sample_submission.csv
Wrote submission.csv with shape: (1856, 4)
Columns: ['record_id', 'encoded_pixels', 'height', 'width']
             record_id                                   encoded_pixels  \
0  4265087624835862793  32126 6 32382 6 32638 6 32894 6 33150 6 33406 6   
1  5940742775168609422  32126 6 32382 6 32638 6 32894 6 33150 6 33406 6   
2    88197744238509538  32126 6 32382 6 32638 6 32894 6 33150 6 33406 6   
3   574607259473142541  32126 6 32382 6 32638 6 32894 6 33150 6 33406 6   
4  1698801436681531032  32126 6 32382 6 32638 6 32894 6 33150 6 33406 6   

   height  width  
0     256    256  
1     256    256  
2     256    256  
3     256    256  
4     256    256  
Unique encoded strings: 1
Encoded_pixels example: 32126 6 32382 6 32638 6 32894 6 33150 6 33406 6
